# QCNN: Hybrid Quantum-Classical CNN for Hyperspectral Nitrogen Classification

**Can Gio Quantum Remote Sensing Project**  
Huu Phuc Le & Minh Tan Le Nguyen

---

This notebook demonstrates the complete two-stage pipeline:

1. **Classical Backbone**: Dual-Branch Inception-ResNet + Gated Attention Fusion -> 112-dim features
2. **Quantum Head**: Fisher -> PCA Whitening -> Dense Angle Encoding -> 8-Qubit VQC -> Classification

**Key Achievement**: 94.50% accuracy with only **99 quantum parameters** (33x fewer than classical 3,267 params)

## Installation & Imports

In [ ]:
# Install dependencies (run once)
# %pip install -r requirements.txt

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import numpy as np
import tensorflow as tf
import torch
import pennylane as qml

print(f"TensorFlow: {tf.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"PennyLane: {qml.__version__}")
print(f"NumPy: {np.__version__}")

# Set seeds for reproducibility
from qcnn.config import SEED
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

## Stage 1: Classical Feature Extraction

Load HSI data, extract patches, train Dual-Branch CNN, extract Attentive_Fusion features

In [ ]:
from qcnn import (
    load_and_preprocess_cube,
    extract_patches,
    create_stratified_splits,
    build_dual_branch_qcnn,
    extract_and_reduce_features,
    SELECTED_BANDS,
    PATCH_SIZE,
    N_CLASSES,
    CLASS_NAMES
)

# ============================================================
# CONFIGURE YOUR DATA PATHS HERE
# ============================================================
DATA_PATH = "path/to/reflectance_260bands-003.npy"      # HSI cube (H, W, 260)
LABEL_PATH = "path/to/Eggplant_Labels.npy"              # Labels (H, W) with 0=bg, 1=Low, 2=Med, 3=High
WAVE_PATH = "path/to/wavelengths_260bands.npy"          # Optional: wavelength array

# Check if data exists
import os
for p, name in [(DATA_PATH, "Data"), (LABEL_PATH, "Labels")]:
    if not os.path.exists(p):
        print(f"WARNING: {name} not found at: {p}")
    else:
        print(f"OK: {name} found: {p}")

In [ ]:
# Load and preprocess HSI cube
print("Loading HSI cube...")
cube, labels = load_and_preprocess_cube(DATA_PATH, LABEL_PATH, WAVE_PATH)
print(f"   Cube shape: {cube.shape} (H, W, {len(SELECTED_BANDS)} bands)")
print(f"   Labels shape: {labels.shape}")
print(f"   Classes: {np.unique(labels[labels >= 0])} -> {CLASS_NAMES}")

In [ ]:
# Extract spatial-spectral patches
print("Extracting patches...")
patches, patch_labels = extract_patches(cube, labels)
print(f"   Patches: {patches.shape}")
print(f"   Labels: {patch_labels.shape}")
print(f"   Class distribution: {np.bincount(patch_labels)}")

In [ ]:
# Stratified train/val/test split
(X_train, y_train), (X_val, y_val), (X_test, y_test) = create_stratified_splits(patches, patch_labels)
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

In [ ]:
# Build and train Dual-Branch CNN
print("Building Dual-Branch Inception-ResNet + Gated Fusion...")
cnn = build_dual_branch_qcnn(n_classes=N_CLASSES)
cnn.summary()

In [ ]:
# Train CNN (uses EarlyStopping & ReduceLROnPlateau)
from qcnn.config import CNN_BATCH_SIZE, CNN_MAX_EPOCHS, CNN_PATIENCE_ES, CNN_PATIENCE_LR

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=CNN_PATIENCE_ES, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=CNN_PATIENCE_LR, min_lr=1e-5),
]

history = cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=CNN_BATCH_SIZE,
    epochs=CNN_MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Extract Attentive_Fusion features (112-dim) + Fisher/PCA reduction to 16-dim quantum-ready
print("Extracting features + Fisher/PCA reduction...")
feat_train_q, feat_val_q, feat_test_q = extract_and_reduce_features(
    cnn, X_train, y_train, X_val, X_test
)
print(f"Train features: {feat_train_q.shape} (should be [N, 16])")
print(f"Val features:   {feat_val_q.shape}")
print(f"Test features:  {feat_test_q.shape}")
print(f"Feature range: [{feat_train_q.min():.4f}, {feat_train_q.max():.4f}] (expected [0, pi])")

## Stage 2: Quantum Classifier Head

Train Classical Dense vs Quantum VQC on identical features for fair comparison

In [ ]:
from qcnn import (
    HybridVQCClassifier,
    ClassicalDenseHead,
    train_model,
    evaluate_model,
    noise_robustness_mc,
    CLASSICAL_LR, CLASSICAL_WEIGHT_DECAY, CLASSICAL_EPOCHS, CLASSICAL_PATIENCE,
    VQC_LR, VQC_EPOCHS, VQC_PATIENCE, VQC_TRAIN_SIZE, VQC_BATCH_SIZE, BATCH_SIZE
)
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

# Prepare PyTorch DataLoaders
train_ds = TensorDataset(torch.tensor(feat_train_q, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(feat_val_q, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(feat_test_q, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

criterion = torch.nn.CrossEntropyLoss()
device = torch.device('cpu')  # Use 'cuda' if available for classical

In [ ]:
# --- Classical Dense Head (3,267 params) ---
print("Training Classical Dense Head...")
classical_model = ClassicalDenseHead(in_features=feat_train_q.shape[1], n_classes=N_CLASSES).to(device)
print(f"   Parameters: {sum(p.numel() for p in classical_model.parameters()):,}")

opt_c = optim.Adam(classical_model.parameters(), lr=CLASSICAL_LR, weight_decay=CLASSICAL_WEIGHT_DECAY)
sched_c = optim.lr_scheduler.ReduceLROnPlateau(opt_c, mode='min', factor=0.5, patience=5)

hist_c, time_c = train_model(
    classical_model, train_loader, val_loader, opt_c, criterion, sched_c,
    epochs=CLASSICAL_EPOCHS, patience=CLASSICAL_PATIENCE, device=device
)

In [ ]:
# --- Quantum VQC Head (99 params) ---
print("Training Quantum VQC Head...")

# Optionally subsample for VQC (config VQC_TRAIN_SIZE = None means full dataset)
if VQC_TRAIN_SIZE is not None and VQC_TRAIN_SIZE < len(feat_train_q):
    idx = np.random.choice(len(feat_train_q), VQC_TRAIN_SIZE, replace=False)
    X_vqc, y_vqc = feat_train_q[idx], y_train[idx]
else:
    X_vqc, y_vqc = feat_train_q, y_train

vqc_train_ds = TensorDataset(torch.tensor(X_vqc, dtype=torch.float32), torch.tensor(y_vqc, dtype=torch.long))
vqc_train_loader = DataLoader(vqc_train_ds, batch_size=VQC_BATCH_SIZE, shuffle=True)

vqc_model = HybridVQCClassifier(n_classes=N_CLASSES).to(device)
print(f"   Parameters: {sum(p.numel() for p in vqc_model.parameters()):,}")

opt_v = optim.Adam(vqc_model.parameters(), lr=VQC_LR)
sched_v = optim.lr_scheduler.ReduceLROnPlateau(opt_v, mode='min', factor=0.5, patience=5)

hist_v, time_v = train_model(
    vqc_model, vqc_train_loader, val_loader, opt_v, criterion, sched_v,
    epochs=VQC_EPOCHS, patience=VQC_PATIENCE, device=device
)

In [ ]:
# --- Evaluation ---
res_c = evaluate_model(classical_model, test_loader, device)
res_v = evaluate_model(vqc_model, test_loader, device)

print("\nTEST RESULTS")
print(f"  Classical: Acc={res_c['accuracy']:.4f} | F1={res_c['f1']:.4f} | Kappa={res_c['kappa']:.4f} | Time={time_c:.1f}s")
print(f"  Quantum:   Acc={res_v['accuracy']:.4f} | F1={res_v['f1']:.4f} | Kappa={res_v['kappa']:.4f} | Time={time_v:.1f}s")

# Per-class F1
from sklearn.metrics import f1_score
f1_c = f1_score(res_c['y_true'], res_c['y_pred'], average=None)
f1_v = f1_score(res_v['y_true'], res_v['y_pred'], average=None)
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: Classical={f1_c[i]:.4f} | Quantum={f1_v[i]:.4f}")

## Noise Robustness Analysis

Monte Carlo evaluation with Gaussian phase jitter sigma in [0.0, 0.5]

In [ ]:
print("Running Monte Carlo noise robustness (5 trials each)...")
noise_summary = noise_robustness_mc(
    {'Classical': classical_model, 'Quantum': vqc_model},
    feat_test_q, y_test
)

print("\nNoise Robustness Summary (Mean +/- Std over 5 trials):")
from qcnn.config import NOISE_LEVELS
for i, sigma in enumerate(NOISE_LEVELS):
    c_mean, c_std = noise_summary['Classical'][0][i], noise_summary['Classical'][1][i]
    q_mean, q_std = noise_summary['Quantum'][0][i], noise_summary['Quantum'][1][i]
    winner = "Classical" if c_mean > q_mean else "Quantum"
    print(f"  sigma={sigma:.2f}: Classical={c_mean:.4f}+/-{c_std:.4f} | Quantum={q_mean:.4f}+/-{q_std:.4f} -> {winner}")

In [ ]:
# Plot noise robustness
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
for name, (means, stds) in noise_summary.items():
    ax.errorbar(NOISE_LEVELS, means, yerr=stds, label=name, marker='o', capsize=4)
ax.set_xlabel('Gaussian Noise sigma')
ax.set_ylabel('Test Accuracy')
ax.set_title('Noise Robustness: Classical vs Quantum VQC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Training Dynamics Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Classical loss
axes[0,0].plot(hist_c['loss'], label='Train')
axes[0,0].plot(hist_c['val_loss'], label='Val')
axes[0,0].set_title('Classical: Loss')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Classical accuracy
axes[0,1].plot(hist_c['accuracy'], label='Train')
axes[0,1].plot(hist_c['val_accuracy'], label='Val')
axes[0,1].set_title('Classical: Accuracy')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Quantum loss
axes[1,0].plot(hist_v['loss'], label='Train')
axes[1,0].plot(hist_v['val_loss'], label='Val')
axes[1,0].set_title('Quantum VQC: Loss')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Quantum accuracy
axes[1,1].plot(hist_v['accuracy'], label='Train')
axes[1,1].plot(hist_v['val_accuracy'], label='Val')
axes[1,1].set_title('Quantum VQC: Accuracy')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Ablation: Quantum Circuit Analysis

In [ ]:
# Inspect the quantum circuit
from qcnn import build_quantum_circuit

qcircuit = build_quantum_circuit()

# Draw circuit (requires matplotlib)
dev = qml.device('default.qubit', wires=8)
@qml.qnode(dev)
def demo_circuit(inputs, weights):
    for i in range(8):
        qml.RY(inputs[i], wires=i)
        qml.RZ(inputs[i+8], wires=i)
    qml.StronglyEntanglingLayers(weights, wires=range(8))
    return [qml.expval(qml.PauliZ(i)) for i in range(8)]

dummy_inputs = np.random.uniform(0, np.pi, 16)
dummy_weights = np.random.uniform(0, 2*np.pi, (3, 8, 3))

fig, ax = qml.draw_mpl(demo_circuit)(dummy_inputs, dummy_weights)
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.show()

## Export Features for Reproducibility

In [ ]:
# Save extracted features for later use
import os
export_dir = "results/features"
os.makedirs(export_dir, exist_ok=True)

np.save(f"{export_dir}/feat_train_q.npy", feat_train_q)
np.save(f"{export_dir}/feat_val_q.npy", feat_val_q)
np.save(f"{export_dir}/feat_test_q.npy", feat_test_q)
np.save(f"{export_dir}/y_train.npy", y_train)
np.save(f"{export_dir}/y_val.npy", y_val)
np.save(f"{export_dir}/y_test.npy", y_test)

print(f"Features saved to {export_dir}/")

## Summary

| Component | Configuration |
|-----------|---------------|
| Input | 11x11x64 patches (64 bands from 260) |
| Classical Backbone | Inception (1x1,3x3,5x5) + ResNet + Gated Fusion |
| Latent Features | 112-dim (Attentive_Fusion) |
| Compression | Fisher Top-64 -> PCA Whitening 16d -> [0,pi] |
| Quantum Encoding | Dense Angle: 2 features/qubit -> 8 qubits |
| VQC Ansatz | StronglyEntanglingLayers x 3 (72 params) |
| Readout | <Z>^8 -> Linear(8->3) + Softmax (27 params) |
| Total Quantum Params | 99 (vs 3,267 classical) |
| Test Accuracy | 94.50% (Quantum) vs 94.48% (Classical) |
| Noise Robustness | Quantum wins for sigma <= 0.20 |

---

Next Steps for Publication/Hardware Deployment:
1. Run on IBM Quantum Heron/Eagle via Qiskit Runtime
2. Apply Zero-Noise Extrapolation (ZNE) & Readout Error Mitigation
3. Extend to rice/soybean HSI datasets for generalization
4. Prepare manuscript for IEEE TGRS / Quantum Science and Technology